In [ ]:
import csv
import gzip
import json
import os
import sys
import random
import bisect
from typing import Dict, Any, List

import numpy as np
import pandas as pd
import optuna

# -----------------------
# Configuration / Hardcoded Paths
# -----------------------
SAMPLES_CSV = "../Data_prep/Sample_csv/processed_training_samples.csv"
OUT_DIR = "training_results_indels"
BLACKLIST_BED = "/Users/marcodupuisrodriguez/Documents/PhD/custom_blacklists/tight_clusters_under_2500bp.bed" 
ARTEFACT_GENES_BED = "/Users/marcodupuisrodriguez/Documents/PhD/custom_blacklists/Artifact_Genes_Blacklist.bed"

N_TRIALS = 2500
SEED = 42

# -----------------------
# Constants
# -----------------------
GENOME_SIZE = 6.4e9
R_CORD = 2e-9
EPS = 1e-15
CORD_WEIGHT = 5.0

# Set seeds
random.seed(SEED)
np.random.seed(SEED)

# -----------------------
# Math Helpers
# -----------------------
def poisson_deviance(y_arr: np.ndarray, lam_arr: np.ndarray, weights: np.ndarray = None) -> float:
    y = np.asarray(y_arr, dtype=float)
    lam = np.asarray(lam_arr, dtype=float) + EPS
    
    with np.errstate(divide='ignore', invalid='ignore'):
        log_term = np.log(y / lam)
        term1 = np.where(y > 0, y * log_term, 0.0)
    
    dev = 2.0 * (term1 - (y - lam))
    if weights is not None:
        dev = dev * weights
        
    return float(np.sum(dev))

def expected_lambda(sample_type: str, age: float, bases: float) -> float:
    st = str(sample_type).lower()
    current_age = float(age) if pd.notna(age) else 0.0
    
    if "cord" in st:
        rate = R_CORD
    elif "pbmc" in st:
        rate = R_CORD + (0.7 * current_age) / GENOME_SIZE
    elif "t_cell" in st or "tcell" in st or "t-cell" in st:
        rate = (14.5 / GENOME_SIZE) + (1.0 * current_age) / GENOME_SIZE
    else:
        # Fallback to the original baseline if an unknown type appears
        rate = R_CORD + (2.0 * current_age) / GENOME_SIZE
        
    return rate * bases

# -----------------------
# BED File Lookup Logic
# -----------------------
class BedLookup:
    def __init__(self, bed_path: str, name: str):
        self.intervals = {} 
        self.name = name
        self._load_bed(bed_path)

    def _load_bed(self, bed_path: str):
        if not os.path.exists(bed_path):
            print(f"[WARN] BED file not found: {bed_path}. {self.name} filter will do nothing.", file=sys.stderr)
            return

        print(f"[INFO] Loading {self.name} from {bed_path}...")
        count = 0
        with open(bed_path, 'r') as f:
            for line in f:
                if line.startswith('#') or not line.strip():
                    continue
                parts = line.strip().split()
                if len(parts) < 3:
                    continue
                chrom, start, end = parts[0], int(parts[1]), int(parts[2])
                
                if chrom not in self.intervals:
                    self.intervals[chrom] = []
                self.intervals[chrom].append((start, end))
                count += 1
        
        for chrom in self.intervals:
            self.intervals[chrom].sort(key=lambda x: x[0])
        print(f"[INFO] Loaded {count} regions for {self.name}.")

    def is_overlapped(self, chrom: str, pos: int) -> bool:
        if chrom not in self.intervals:
            return False
        
        regions = self.intervals[chrom]
        query_idx = pos - 1 
        idx = bisect.bisect_right(regions, (query_idx, float('inf')))
        
        for i in range(max(0, idx - 1), -1, -1):
            start, end = regions[i]
            if end <= query_idx:
                break 
            if start <= query_idx < end:
                return True
        return False

# -----------------------
# VCF Parsing to DataFrame
# -----------------------
def parse_vcf_to_df(vcf_path: str, sample_id: str) -> pd.DataFrame:
    data = []
    if not os.path.exists(vcf_path):
        print(f"[WARN] File not found: {vcf_path}", file=sys.stderr)
        return pd.DataFrame()

    openf = gzip.open if vcf_path.endswith(".gz") else open
    try:
        with openf(vcf_path, "rt") as fh:
            for line in fh:
                if line.startswith("#"):
                    continue
                fields = line.strip().split("\t")
                chrom, pos = fields[0], int(fields[1])
                alt = fields[4] # NEW: Extract the ALT sequence
                info_str = fields[7]
                info_dict = {k: v for item in info_str.split(";") for k, v in [item.split("=", 1) if "=" in item else (item, "Yes")]}
                
                # NEW: Added ALT to the dictionary
                info_dict.update({"sample_id": sample_id, "CHROM": chrom, "POS": pos, "ALT": alt})
                data.append(info_dict)
    except Exception as e:
        print(f"[ERROR] Error reading {vcf_path}: {e}", file=sys.stderr)
        return pd.DataFrame()

    if not data:
        return pd.DataFrame()

    df = pd.DataFrame(data)
    
    def to_num(col, dtype, default):
        return pd.to_numeric(df[col], errors='coerce').fillna(default).astype(dtype) if col in df.columns else pd.Series([default]*len(df), dtype=dtype)

    def to_str(col, default):
        return df[col].fillna(default) if col in df.columns else pd.Series([default]*len(df))

    df['CHROM'] = to_str('CHROM', 'chr1')
    df['ALT'] = to_str('ALT', '') # NEW: String column for ALT
    df['POS'] = to_num('POS', int, 0)
    df['INDEL_LENGTH'] = to_num('INDEL_LENGTH', int, 0)
    df['MEAN_INSET_QUAL'] = to_num('MEAN_INSET_QUAL', float, -1.0)
    df['MEAN_DEL_ANCHOR_QUAL'] = to_num('MEAN_DEL_ANCHOR_QUAL', float, -1.0)
    df['HOMOPOLYMER_LEN'] = to_num('HOMOPOLYMER_LEN', int, 0)
    df['N_SLIPPAGE_SSCS'] = to_num('N_SLIPPAGE_SSCS', int, 0)
    df['GC_CONTENT'] = to_num('GC_CONTENT', float, 0.5) 
    df['DIST_TO_OTHER_INDEL'] = to_num('DIST_TO_OTHER_INDEL', int, 10000)
    df['RP_MED_SSCS'] = to_num('RP_MED_SSCS', float, 75.0)
    df['AVG_ALT_N_COUNT'] = to_num('AVG_ALT_N_COUNT', float, 0.0)
    df['N_ALT_SSCS'] = to_num('N_ALT_SSCS', int, 0)
    df['N_TOTAL'] = to_num('N_TOTAL', int, 0)
    df['N_ALT'] = to_num('N_ALT', int, 0) 
    df['N_TOTAL_SSCS'] = to_num('N_TOTAL_SSCS', int, -1)
    df['DEPTH_PERCENTILE'] = to_num('DEPTH_PERCENTILE', int, 50)
    df['VARIANTS_20BP'] = to_num('VARIANTS_20BP', int, 0)
    df['VARIANTS_250BP'] = to_num('VARIANTS_250BP', int, 0)
    df['MEDIAN_DIST_TO_SOFTCLIP'] = to_num('MEDIAN_DIST_TO_SOFTCLIP', int, 1000)
    df['AVG_ASXS'] = to_num('AVG_ALT_ASXS', float, 0.0) 
    df['MEAN_MAPQ'] = to_num('MEAN_ALT_MAPQ', float, 0.0) 
    df['AVG_ALT_EXCESS_NM'] = to_num('AVG_ALT_EXCESS_NM', float, 0.0)
    df['AVG_ALT_SOFTCLIP'] = to_num('AVG_ALT_SOFTCLIP', float, 0.0)
    df['REF_ENTROPY_40BP'] = to_num('REF_ENTROPY_40BP', float, 2.0)
    df['IN_BLACKLIST'] = to_str('IN_BLACKLIST', "NO")
    df['IN_GNOMAD_COMMON'] = to_str('IN_GNOMAD_COMMON', "No")
    df['IN_CB_PON'] = to_str('IN_CB_PON', "No")

    with np.errstate(divide='ignore', invalid='ignore'):
        df['VAF_SSCS'] = (df['N_ALT_SSCS'] / df['N_TOTAL_SSCS']).fillna(0.0)

    return df

# -----------------------
# Aggregation & Filtering Logic
# -----------------------
def aggregate_samples(samples_meta: List[Dict]) -> List[Dict]:
    """Groups all cord blood samples to combat Poisson noise."""
    agg_meta, cord_ids = [], []
    cord_exp, cord_bases = 0.0, 0.0
    
    for s in samples_meta:
        lam = expected_lambda(s['sample_type'], s['age'], s['bases'])
        if "cord" in s['sample_type'].lower():
            cord_ids.append(s['sample_id'])
            cord_exp += lam
            cord_bases += s['bases']
        else:
            agg_meta.append({
                'sample_ids': [s['sample_id']],
                'sample_type': s['sample_type'],
                'age': s['age'],
                'bases': s['bases'],
                'expected_lambda': lam,
                'is_cord': False
            })
            
    if cord_ids:
        agg_meta.append({
            'sample_ids': cord_ids, 
            'sample_type': 'cord_aggregate',
            'age': 0.0, 
            'bases': cord_bases,
            'expected_lambda': cord_exp,
            'is_cord': True
        })
    return agg_meta

def get_static_mask(df: pd.DataFrame, pon_flags: Dict[str, bool], ignore_blacklists: bool = False, ignore_cb_pon: bool = False) -> pd.Series:
    """Filters evaluated ONCE before Optuna runs to save immense computation time."""
    mask = (df['N_TOTAL_SSCS'] > 0) & (df['N_ALT_SSCS'] > 1)
    
    # Filter out insertions that contain 'N' in the inserted sequence
    has_n_in_insertion = (df['INDEL_LENGTH'] > 0) & (df['ALT'].str.contains('N', case=False, na=False))
    mask &= (~has_n_in_insertion)
    
    if pon_flags.get("use_cord_pon", False):
        mask &= (df['IN_BLACKLIST'] != "YES")

    # NEW: Hard filter for IN_CB_PON
    if not ignore_cb_pon:
        mask &= (df['IN_CB_PON'] != "Yes")

    if not ignore_blacklists:
        mask &= (~df['IN_CLUSTER_BLACKLIST']) & (~df['IN_ARTEFACT_BLACKLIST'])
        
    return mask

def get_dynamic_mask(df: pd.DataFrame, thresholds: Dict[str, Any]) -> pd.Series:
    """Filters evaluated during every Optuna trial."""
    mask = (df['N_TOTAL'] >= thresholds['n_total_min'])
    mask &= (df['DEPTH_PERCENTILE'] <= thresholds['depth_percentile_max'])
    mask &= (df['MEAN_MAPQ'] >= thresholds['mean_mapq_min'])
    mask &= (df['AVG_ASXS'] > thresholds['asxs_min'])

    is_ins = df['INDEL_LENGTH'] > 0
    is_del = df['INDEL_LENGTH'] < 0
    mask &= ((is_ins & (df['MEAN_INSET_QUAL'] >= thresholds['mean_inset_qual_min'])) | 
             (is_del & (df['MEAN_DEL_ANCHOR_QUAL'] >= thresholds['mean_del_anchor_qual_min'])))

    is_1bp = df['INDEL_LENGTH'].abs() == 1
    is_long = df['INDEL_LENGTH'].abs() > 1
    mask &= ((is_1bp & (df['HOMOPOLYMER_LEN'] <= thresholds['hp_max_1bp'])) | 
             (is_long & (df['HOMOPOLYMER_LEN'] <= thresholds['hp_max_long'])))
    
    mask &= (df['N_SLIPPAGE_SSCS'] <= thresholds['max_slippage_reads'])
    mask &= (df['VARIANTS_20BP'] <= thresholds['variants_20bp_max'])
    mask &= (df['VARIANTS_250BP'] <= thresholds['variants_250bp_max'])
    mask &= (df['REF_ENTROPY_40BP'] >= thresholds['ref_entropy_min'])
    mask &= (df['AVG_ALT_SOFTCLIP'] <= thresholds['avg_alt_softclip_max'])
    mask &= (df['MEDIAN_DIST_TO_SOFTCLIP'] >= thresholds['dist_to_softclip_min'])
    mask &= (df['AVG_ALT_EXCESS_NM'] <= thresholds['avg_alt_nm_max'])
    mask &= (df['GC_CONTENT'] >= thresholds['gc_min']) & (df['GC_CONTENT'] <= thresholds['gc_max'])
    mask &= (df['DIST_TO_OTHER_INDEL'] >= thresholds['min_dist_to_indel'])
    mask &= (df['RP_MED_SSCS'] >= thresholds['min_rp_med']) & (df['RP_MED_SSCS'] <= thresholds['max_rp_med'])
    mask &= (df['AVG_ALT_N_COUNT'] <= thresholds['max_n_count'])
    mask &= (df['VAF_SSCS'] <= thresholds['germline_vaf_max'])
    return mask

# -----------------------
# Optimization Objective
# -----------------------
def make_objective(filtered_df: pd.DataFrame, agg_meta: List[Dict]):
    exp_arr = np.array([m['expected_lambda'] for m in agg_meta])
    weights_arr = np.array([CORD_WEIGHT if m['is_cord'] else 1.0 for m in agg_meta])

    def objective(trial):
        thresholds = {
            "n_total_min": trial.suggest_int("n_total_min", 15, 30),
            "depth_percentile_max": trial.suggest_int("depth_percentile_max", 95, 99),
            "mean_inset_qual_min": trial.suggest_int("mean_inset_qual_min", 20, 50),
            "mean_del_anchor_qual_min": trial.suggest_int("mean_del_anchor_qual_min", 20, 50),
            "hp_max_1bp": trial.suggest_int("hp_max_1bp", 3, 6),
            "hp_max_long": trial.suggest_int("hp_max_long", 8, 15),
            "max_slippage_reads": trial.suggest_int("max_slippage_reads", 0, 5),
            "asxs_min": trial.suggest_int("asxs_min", 10, 60),
            "mean_mapq_min": trial.suggest_int("mean_mapq_min", 20, 50),
            "avg_alt_nm_max": trial.suggest_float("avg_alt_nm_max", 0.0, 5.0),
            "avg_alt_softclip_max": trial.suggest_float("avg_alt_softclip_max", 0.0, 5.0),
            "dist_to_softclip_min": trial.suggest_int("dist_to_softclip_min", 0, 5),
            "variants_20bp_max": trial.suggest_int("variants_20bp_max", 0, 4),
            "variants_250bp_max": trial.suggest_int("variants_250bp_max", 0, 15),
            "ref_entropy_min": trial.suggest_float("ref_entropy_min", 0.0, 2.1),
            "gc_min": trial.suggest_float("gc_min", 0.01, 0.2),
            "gc_max": trial.suggest_float("gc_max", 0.8, 0.99),
            "min_dist_to_indel": trial.suggest_int("min_dist_to_indel", 0, 500),
            "min_rp_med": trial.suggest_float("min_rp_med", 0.01, 0.08),
            "max_rp_med": trial.suggest_float("max_rp_med", 0.92, 0.99),
            "max_n_count": trial.suggest_float("max_n_count", 0.0, 3.0),
            "germline_vaf_max": trial.suggest_float("germline_vaf_max", 0.25, 0.35),
        }

        dyn_mask = get_dynamic_mask(filtered_df, thresholds)
        obs_counts = filtered_df[dyn_mask]['sample_id'].value_counts()
        
        obs_arr = [sum(obs_counts.get(sid, 0) for sid in m['sample_ids']) for m in agg_meta]
        return poisson_deviance(np.array(obs_arr), exp_arr, weights=weights_arr)
    
    return objective

# -----------------------
# Reporting
# -----------------------
def evaluate_and_report(best_thresholds, pon_flags, all_variants_df, samples_meta, outdir):
    os.makedirs(outdir, exist_ok=True)
    
    # NEW: Added ignore_cb_pon flags to isolate the counts
    static_mask_final = get_static_mask(all_variants_df, pon_flags, ignore_blacklists=False, ignore_cb_pon=False)
    static_mask_clean = get_static_mask(all_variants_df, pon_flags, ignore_blacklists=True, ignore_cb_pon=False)
    static_mask_no_cb_pon = get_static_mask(all_variants_df, pon_flags, ignore_blacklists=False, ignore_cb_pon=True)
    
    dyn_mask = get_dynamic_mask(all_variants_df, best_thresholds)
    
    pass_mask_final = static_mask_final & dyn_mask
    pass_mask_clean = static_mask_clean & dyn_mask
    pass_mask_no_cb_pon = static_mask_no_cb_pon & dyn_mask # NEW
    
    count_saved_by_blacklist = (pass_mask_clean & (~pass_mask_final)).sum()
    count_saved_by_cb_pon = (pass_mask_no_cb_pon & (~pass_mask_final)).sum() # NEW
    
    obs_series = all_variants_df[pass_mask_final].groupby('sample_id').size()
    
    observed, expected, ids, ages, bases, types, weights = [], [], [], [], [], [], []

    for s in samples_meta:
        sid = s['sample_id']
        ids.append(sid)
        observed.append(obs_series.get(sid, 0))
        expected.append(expected_lambda(s['sample_type'], s['age'], s['bases']))
        ages.append(s['age'])
        bases.append(s['bases'])
        types.append(s['sample_type'])
        weights.append(CORD_WEIGHT if "cord" in s['sample_type'].lower() else 1.0)

    obs_arr, exp_arr, weights_arr = np.array(observed), np.array(expected), np.array(weights)

    with open(os.path.join(outdir, "best_thresholds.json"), "w") as fh:
        json.dump({"thresholds": best_thresholds, "pon_flags": pon_flags, "blacklists_enforced": True, "cord_penalty_weight": CORD_WEIGHT}, fh, indent=2)

    dev = poisson_deviance(obs_arr, exp_arr, weights=weights_arr)
    mean_ratio = float(np.mean(obs_arr / (exp_arr + EPS)))
    
    with open(os.path.join(outdir, "summary.json"), "w") as fh:
        json.dump({
            "n_samples": len(samples_meta),
            "obs_mean_over_expected": mean_ratio,
            "total_observed": int(np.sum(obs_arr)),
            "total_expected": float(np.sum(exp_arr)),
            "weighted_poisson_deviance": dev,
            "variants_blocked_by_blacklists": int(count_saved_by_blacklist),
            "variants_blocked_by_cb_pon": int(count_saved_by_cb_pon) # NEW
        }, fh, indent=2)

    pd.DataFrame({
        "sample_id": ids, "sample_type": types, "age": ages, "bases": bases,
        "observed": observed, "expected": expected, "weight_used": weights 
    }).to_csv(os.path.join(outdir, "observed_expected.csv"), index=False)
    
    final_pass_df = all_variants_df[pass_mask_final]
    n_ins = (final_pass_df['INDEL_LENGTH'] > 0).sum()
    n_del = (final_pass_df['INDEL_LENGTH'] < 0).sum()
    n_1bp = (final_pass_df['INDEL_LENGTH'].abs() == 1).sum()
    n_long = (final_pass_df['INDEL_LENGTH'].abs() > 1).sum()
    
    ins_del_ratio = n_ins / n_del if n_del > 0 else 0.0
    bp1_long_ratio = n_1bp / n_long if n_long > 0 else 0.0

    print("-" * 30)
    print("FINAL RESULTS")
    print("-" * 30)
    print(f"Total Samples:         {len(samples_meta)}")
    print(f"Total Observed Indels: {int(np.sum(obs_arr))}")
    print(f"Total Expected Indels: {float(np.sum(exp_arr)):.2f}")
    print(f"Weighted Deviance:     {dev:.4f}")
    print(f"Mean Obs/Exp Ratio:    {mean_ratio:.4f}")
    print(f"Blacklist Blocked:     {count_saved_by_blacklist}")
    print(f"CB PON Blocked:        {count_saved_by_cb_pon}") # NEW
    print("-" * 30)
    print("INDEL METRICS (Passing Variants)")
    print("-" * 30)
    print(f"Insertions: {n_ins} | Deletions: {n_del}")
    print(f"INS / DEL Ratio:       {ins_del_ratio:.3f}")
    print(f"1bp Indels: {n_1bp} | >1bp Indels: {n_long}")
    print(f"1bp / >1bp Ratio:      {bp1_long_ratio:.3f}")
    print("-" * 30)

    
# -----------------------
# Execution Block
# -----------------------
if __name__ == "__main__":
    print(f"[INFO] Reading metadata from {SAMPLES_CSV}...")
    samples_meta = []
    with open(SAMPLES_CSV, newline='') as csvfile:
        for r in csv.DictReader(csvfile):
            samples_meta.append({
                "sample_id": r["sample_id"],
                "vcf_path": r["vcf_path"],
                "sample_type": r["sample_type"].lower(),
                "age": float(r["age"]) if r.get("age") else None,
                "bases": float(r["bases"]),
            })
    print(f"[INFO] Loaded {len(samples_meta)} samples.")

    print("[INFO] Loading VCFs into DataFrame...")
    df_list = [parse_vcf_to_df(s["vcf_path"], s["sample_id"]) for s in samples_meta]
    all_variants_df = pd.concat(df_list, ignore_index=True) if df_list else pd.DataFrame()
    
    print(f"[INFO] Total variants loaded: {len(all_variants_df)}")

    if not all_variants_df.empty:
        chroms = all_variants_df['CHROM'].values
        poses = all_variants_df['POS'].values

        bed_lookup_cluster = BedLookup(BLACKLIST_BED, "Cluster Blacklist")
        # Optimized list comprehension replacing df.apply
        all_variants_df['IN_CLUSTER_BLACKLIST'] = [bed_lookup_cluster.is_overlapped(c, p) for c, p in zip(chroms, poses)]
        
        bed_lookup_genes = BedLookup(ARTEFACT_GENES_BED, "Artefact Gene Blacklist")
        all_variants_df['IN_ARTEFACT_BLACKLIST'] = [bed_lookup_genes.is_overlapped(c, p) for c, p in zip(chroms, poses)]
        
        hits_cluster = sum(all_variants_df['IN_CLUSTER_BLACKLIST'])
        print(f"[INFO] Found {hits_cluster} variants in Cluster blacklist.")

        # Aggregate cords for robust optimization
        agg_meta = aggregate_samples(samples_meta)
        
        # Pre-filter static masks to massively speed up Optuna
        pon_flags = {"use_cord_pon": True}
        static_mask = get_static_mask(all_variants_df, pon_flags, ignore_blacklists=False)
        static_filtered_df = all_variants_df[static_mask].copy()

        print(f"[INFO] Starting Optuna (Trials={N_TRIALS}). Evaluating {len(static_filtered_df)} valid candidates per trial...")
        study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=SEED))
        
        objective = make_objective(static_filtered_df, agg_meta)
        study.optimize(objective, n_trials=N_TRIALS)

        print(f"[INFO] Best Weighted Deviance: {study.best_value:.4f}")

        print("[INFO] Generating report...")
        evaluate_and_report(study.best_params, pon_flags, all_variants_df, samples_meta, OUT_DIR)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

# -----------------------
# Configuration
# -----------------------
CSV_PATH = "training_results_indels/observed_expected.csv"
OUTPUT_PNG = "training_results_indels/train_observed_vs_expected_plot.png"

# Check if file exists
if not os.path.exists(CSV_PATH):
    print(f"[ERROR] Could not find {CSV_PATH}. Please run the optimization cell first.")
else:
    # Load table
    df = pd.read_csv(CSV_PATH)

    # -----------------------
    # Data Processing & Aggregation
    # -----------------------
    # Identify cord blood samples
    is_cord = df['sample_type'].str.lower().str.contains('cord')
    
    df_cord = df[is_cord]
    df_other = df[~is_cord]

    # Calculate rates for non-cord samples
    ages = list(df_other["age"].values)
    obs_rates = list(df_other["observed"].values / df_other["bases"].values)
    exp_rates = list(df_other["expected"].values / df_other["bases"].values)

    # Aggregate cord samples if they exist
    if not df_cord.empty:
        total_cord_obs = df_cord["observed"].sum()
        total_cord_exp = df_cord["expected"].sum()
        total_cord_bases = df_cord["bases"].sum()
        
        # Append the aggregated cord blood point
        ages.append(0.0) # Cord blood is age 0
        obs_rates.append(total_cord_obs / total_cord_bases)
        exp_rates.append(total_cord_exp / total_cord_bases)

    # Convert back to numpy arrays for plotting
    ages = np.array(ages)
    obs_rates = np.array(obs_rates)
    exp_rates = np.array(exp_rates)

    # -----------------------
    # Plotting
    # -----------------------
    fig, ax = plt.subplots(figsize=(8, 6))

    # Plot Observed (Blue circles)
    ax.scatter(ages, obs_rates, label="Observed", color="#1f77b4", alpha=0.8, marker="o", s=60, edgecolors='k', linewidth=0.5)

    # Plot Expected (Orange crosses)
    ax.scatter(ages, exp_rates, label="Expected", color="#ff7f0e", alpha=0.8, marker="x", s=60, linewidth=2)

    # Styling
    ax.set_xlabel("Age (years)", fontsize=12)
    ax.set_ylabel("Indel rate (Indels per base)", fontsize=12) # Updated to Indel
    
    ax.set_xlim(-2, 100)
    ax.set_ylim(0, 2e-8)
    
    ax.set_title("Observed vs Expected Indel Rate (Training Set)", fontsize=14) # Updated to Indel
    ax.grid(True, which='both', linestyle='--', linewidth=0.5, alpha=0.5)
    
    # Legend
    ax.legend(loc='upper left', frameon=True, fontsize=10)

    plt.tight_layout()
    
    # Save and Show
    plt.savefig(OUTPUT_PNG, dpi=150)
    print(f"[INFO] Plot saved to: {OUTPUT_PNG}")
    plt.show()